# EDA Tổng Quan Dữ Liệu E-commerce Clickstream

Notebook này dùng để tìm hiểu dữ liệu ban đầu theo hướng dễ đọc và có thể dùng lại cho báo cáo. Nội dung được chia thành 5 phần:

1. Giới thiệu và khởi tạo
2. Kiểm kê cấu trúc dữ liệu thô
3. Đánh giá chất lượng dữ liệu
4. Kiểm tra toàn vẹn quan hệ
5. Phân tích phân phối và nghiệp vụ cơ bản

Quy ước sử dụng trong notebook:

- Raw data dùng để hiểu dữ liệu gốc trước khi cleaning.
- Processed data dùng để kiểm tra quan hệ sau khi Task 1 đã chuẩn hóa dữ liệu.
- Không chỉnh sửa trực tiếp dữ liệu raw trong notebook này.

## 1. Giới thiệu và khởi tạo

Phần này khởi tạo môi trường chạy notebook, nạp cấu hình đường dẫn từ project, tạo SparkSession và đọc toàn bộ 7 file CSV raw.

In [1]:
from pathlib import Path
import sys

# Tìm thư mục gốc của project bằng cách đi ngược từ vị trí hiện tại cho đến khi thấy thư mục src.
thu_muc_hien_tai = Path.cwd().resolve()
cac_thu_muc_can_kiem_tra = [thu_muc_hien_tai] + list(thu_muc_hien_tai.parents)
thu_muc_du_an = next(
    thu_muc
    for thu_muc in cac_thu_muc_can_kiem_tra
    if (thu_muc / "src" / "common" / "config.py").exists()
)

# Thêm thư mục src vào Python path để notebook dùng lại config và helper có sẵn của dự án.
duong_dan_src = thu_muc_du_an / "src"
if str(duong_dan_src) not in sys.path:
    sys.path.insert(0, str(duong_dan_src))

from common.config import RAW_DIR, PROCESSED_DIR, RAW_TABLE_NAMES
from common.spark_utils import create_spark_session

# Tạo SparkSession dùng chung cho toàn bộ notebook.
spark = create_spark_session("eda-overview-ecommerce-clickstream")

print(f"Thu muc du an: {thu_muc_du_an}")
print(f"Thu muc raw data: {RAW_DIR}")
print(f"Thu muc processed data: {PROCESSED_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 19:03:59 WARN Utils: Your hostname, linhtd3993-Katana-15-B13UDXK, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface wlo1)
26/06/13 19:03:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 19:04:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Thu muc du an: /home/linhtd3993/workspace/projects/Big_data/ecommerce_clickstream_spark
Thu muc raw data: /home/linhtd3993/workspace/projects/Big_data/ecommerce_clickstream_spark/data/raw/kaggle_original
Thu muc processed data: /home/linhtd3993/workspace/projects/Big_data/ecommerce_clickstream_spark/data/processed


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Đọc một bảng raw CSV bằng Spark với inferSchema để quan sát kiểu dữ liệu Spark tự nhận diện.
def doc_bang_raw(ten_bang):
    duong_dan_bang = RAW_DIR / f"{ten_bang}.csv"
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(duong_dan_bang))
    )

# Đọc toàn bộ 7 bảng raw vào một dictionary để các phần sau dùng lại.
du_lieu_raw = {
    ten_bang: doc_bang_raw(ten_bang)
    for ten_bang in RAW_TABLE_NAMES
}

print("Da doc xong cac bang raw:")
print(list(du_lieu_raw.keys()))

Da doc xong cac bang raw:
['customers', 'sessions', 'events', 'products', 'orders', 'order_items', 'reviews']


## 2. Kiểm kê cấu trúc dữ liệu thô

Phần này trả lời các câu hỏi cơ bản: mỗi bảng có bao nhiêu dòng, bao nhiêu cột, schema Spark tự suy diễn là gì, và 5 dòng đầu tiên trông như thế nào.

In [3]:
# Đếm số dòng và số cột của từng bảng để có bảng inventory tổng quan.
thong_tin_cau_truc = []

for ten_bang, bang_du_lieu in du_lieu_raw.items():
    so_dong = bang_du_lieu.count()
    so_cot = len(bang_du_lieu.columns)
    thong_tin_cau_truc.append((ten_bang, so_dong, so_cot, ", ".join(bang_du_lieu.columns)))

bang_cau_truc = spark.createDataFrame(
    thong_tin_cau_truc,
    ["ten_bang", "so_dong", "so_cot", "danh_sach_cot"]
)

bang_cau_truc.orderBy("ten_bang").show(truncate=False)

+-----------+-------+------+-----------------------------------------------------------------------------------------------------------------+
|ten_bang   |so_dong|so_cot|danh_sach_cot                                                                                                    |
+-----------+-------+------+-----------------------------------------------------------------------------------------------------------------+
|customers  |20000  |7     |customer_id, name, email, country, age, signup_date, marketing_opt_in                                            |
|events     |760958 |10    |event_id, session_id, timestamp, event_type, product_id, qty, cart_size, payment, discount_pct, amount_usd       |
|order_items|59163  |5     |order_id, product_id, unit_price_usd, quantity, line_total_usd                                                   |
|orders     |33580  |10    |order_id, customer_id, order_time, payment_method, discount_pct, subtotal_usd, total_usd, country, device, source|

In [4]:
# In schema của từng bảng để kiểm tra kiểu dữ liệu mà Spark tự suy diễn từ raw CSV.
for ten_bang, bang_du_lieu in du_lieu_raw.items():
    print(f"\n===== SCHEMA: {ten_bang} =====")
    bang_du_lieu.printSchema()


===== SCHEMA: customers =====
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- marketing_opt_in: boolean (nullable = true)


===== SCHEMA: sessions =====
root
 |-- session_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- device: string (nullable = true)
 |-- source: string (nullable = true)
 |-- country: string (nullable = true)


===== SCHEMA: events =====
root
 |-- event_id: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- cart_size: double (nullable = true)
 |-- payment: string (nullable = true)
 |-- discount_pct: double (nullable = 

In [5]:
# Hiển thị 5 dòng đầu tiên của từng bảng để quan sát trực tiếp dữ liệu gốc.
for ten_bang, bang_du_lieu in du_lieu_raw.items():
    print(f"\n===== SAMPLE: {ten_bang} =====")
    bang_du_lieu.show(5, truncate=False)


===== SAMPLE: customers =====
+-----------+----------------+--------------------------+-------+---+-----------+----------------+
|customer_id|name            |email                     |country|age|signup_date|marketing_opt_in|
+-----------+----------------+--------------------------+-------+---+-----------+----------------+
|1          |Jennifer Salinas|nicholas59@example.org    |JP     |71 |2020-09-04 |true            |
|2          |Phillip Ramos   |christinarubio@example.com|IN     |26 |2020-04-05 |false           |
|3          |Dawn Fowler     |jessica03@example.org     |BR     |21 |2023-08-31 |true            |
|4          |Mario Butler    |paula27@example.org       |FR     |63 |2022-06-30 |true            |
|5          |Amber Brown     |kevin85@example.net       |BR     |19 |2022-07-22 |true            |
+-----------+----------------+--------------------------+-------+---+-----------+----------------+
only showing top 5 rows

===== SAMPLE: sessions =====
+----------+-----------+

## 3. Đánh giá chất lượng dữ liệu

Phần này kiểm tra null, duplicate và tính nhất quán của các trường phân loại. Với bảng `events`, null cần được phân tích theo `event_type` vì một số null là hợp lệ theo nghiệp vụ.

In [6]:
# Tính số lượng null và tỷ lệ null trên từng cột của một bảng.
def tinh_null_theo_cot(ten_bang, bang_du_lieu):
    so_dong = bang_du_lieu.count()
    bieu_thuc_null = [
        F.sum(F.when(F.col(ten_cot).isNull(), 1).otherwise(0)).alias(ten_cot)
        for ten_cot in bang_du_lieu.columns
    ]
    dong_null = bang_du_lieu.agg(*bieu_thuc_null).collect()[0].asDict()
    ket_qua = [
        (
            ten_bang,
            ten_cot,
            int(so_luong_null),
            round(float(so_luong_null) * 100 / so_dong, 4) if so_dong > 0 else 0.0,
        )
        for ten_cot, so_luong_null in dong_null.items()
    ]
    return spark.createDataFrame(ket_qua, ["ten_bang", "ten_cot", "so_luong_null", "ty_le_null_phan_tram"])

# Ghép kết quả null của tất cả bảng thành một bảng duy nhất để dễ đọc.
cac_bang_null = [
    tinh_null_theo_cot(ten_bang, bang_du_lieu)
    for ten_bang, bang_du_lieu in du_lieu_raw.items()
]

bang_null_tong_hop = cac_bang_null[0]
for bang_null in cac_bang_null[1:]:
    bang_null_tong_hop = bang_null_tong_hop.unionByName(bang_null)

bang_null_tong_hop.orderBy("ten_bang", F.desc("ty_le_null_phan_tram")).show(200, truncate=False)

+-----------+----------------+-------------+--------------------+
|ten_bang   |ten_cot         |so_luong_null|ty_le_null_phan_tram|
+-----------+----------------+-------------+--------------------+
|customers  |email           |0            |0.0                 |
|customers  |customer_id     |0            |0.0                 |
|customers  |marketing_opt_in|0            |0.0                 |
|customers  |signup_date     |0            |0.0                 |
|customers  |country         |0            |0.0                 |
|customers  |age             |0            |0.0                 |
|customers  |name            |0            |0.0                 |
|events     |amount_usd      |727378       |95.5871             |
|events     |payment         |727378       |95.5871             |
|events     |discount_pct    |727378       |95.5871             |
|events     |cart_size       |716049       |94.0984             |
|events     |qty             |617832       |81.1913             |
|events   

In [7]:
# Phân tích null của events theo event_type để phân biệt null do nghiệp vụ và null do lỗi dữ liệu.
bang_events = du_lieu_raw["events"]
cac_cot_can_kiem_tra_null_events = ["product_id", "qty", "cart_size", "payment", "discount_pct", "amount_usd"]

bieu_thuc_null_theo_event = []
for ten_cot in cac_cot_can_kiem_tra_null_events:
    bieu_thuc_null_theo_event.append(
        F.sum(F.when(F.col(ten_cot).isNull(), 1).otherwise(0)).alias(f"{ten_cot}_null")
    )
    bieu_thuc_null_theo_event.append(
        F.round(
            F.sum(F.when(F.col(ten_cot).isNull(), 1).otherwise(0)) * 100 / F.count("*"),
            4,
        ).alias(f"{ten_cot}_null_pct")
    )

bang_null_events_theo_loai = (
    bang_events
    .groupBy("event_type")
    .agg(F.count("*").alias("so_dong"), *bieu_thuc_null_theo_event)
    .orderBy("event_type")
)

bang_null_events_theo_loai.show(truncate=False)

+-----------+-------+---------------+-------------------+--------+------------+--------------+------------------+------------+----------------+-----------------+---------------------+---------------+-------------------+
|event_type |so_dong|product_id_null|product_id_null_pct|qty_null|qty_null_pct|cart_size_null|cart_size_null_pct|payment_null|payment_null_pct|discount_pct_null|discount_pct_null_pct|amount_usd_null|amount_usd_null_pct|
+-----------+-------+---------------+-------------------+--------+------------+--------------+------------------+------------+----------------+-----------------+---------------------+---------------+-------------------+
|add_to_cart|143126 |0              |0.0                |0       |0.0         |143126        |100.0             |143126      |100.0           |143126           |100.0                |143126         |100.0              |
|checkout   |44909  |44909          |100.0              |44909   |100.0       |0             |0.0               |44909  

In [8]:
# Đếm số dòng trùng lặp hoàn toàn bằng cách lấy tổng số dòng trừ số dòng distinct.
def dem_trung_lap_hoan_toan(ten_bang, bang_du_lieu):
    so_dong = bang_du_lieu.count()
    so_dong_khac_nhau = bang_du_lieu.distinct().count()
    return (ten_bang, so_dong - so_dong_khac_nhau)

# Đếm số bản ghi thuộc nhóm trùng khóa chính để biết khóa định danh có bị lặp hay không.
def dem_trung_lap_khoa_chinh(ten_bang, bang_du_lieu, cac_cot_khoa_chinh):
    bang_trung_khoa = (
        bang_du_lieu
        .groupBy(*cac_cot_khoa_chinh)
        .count()
        .filter(F.col("count") > 1)
    )
    so_nhom_trung = bang_trung_khoa.count()
    so_ban_ghi_trong_nhom_trung = bang_trung_khoa.agg(F.sum("count").alias("tong")).collect()[0]["tong"]
    return (ten_bang, ", ".join(cac_cot_khoa_chinh), so_nhom_trung, int(so_ban_ghi_trong_nhom_trung or 0))

# Khai báo khóa chính theo schema mapping của dự án.
khoa_chinh_theo_bang = {
    "events": ["event_id"],
    "customers": ["customer_id"],
    "sessions": ["session_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "product_id"],
    "reviews": ["review_id"],
}

ket_qua_trung_lap_hoan_toan = [
    dem_trung_lap_hoan_toan(ten_bang, bang_du_lieu)
    for ten_bang, bang_du_lieu in du_lieu_raw.items()
]

ket_qua_trung_lap_khoa_chinh = [
    dem_trung_lap_khoa_chinh(ten_bang, du_lieu_raw[ten_bang], cac_cot_khoa_chinh)
    for ten_bang, cac_cot_khoa_chinh in khoa_chinh_theo_bang.items()
]

spark.createDataFrame(
    ket_qua_trung_lap_hoan_toan,
    ["ten_bang", "so_dong_trung_lap_hoan_toan"]
).orderBy("ten_bang").show(truncate=False)

spark.createDataFrame(
    ket_qua_trung_lap_khoa_chinh,
    ["ten_bang", "khoa_chinh", "so_nhom_trung_khoa", "so_ban_ghi_trong_nhom_trung"]
).orderBy("ten_bang").show(truncate=False)

+-----------+---------------------------+
|ten_bang   |so_dong_trung_lap_hoan_toan|
+-----------+---------------------------+
|customers  |0                          |
|events     |0                          |
|order_items|73                         |
|orders     |0                          |
|products   |0                          |
|reviews    |0                          |
|sessions   |0                          |
+-----------+---------------------------+

+-----------+--------------------+------------------+---------------------------+
|ten_bang   |khoa_chinh          |so_nhom_trung_khoa|so_ban_ghi_trong_nhom_trung|
+-----------+--------------------+------------------+---------------------------+
|customers  |customer_id         |0                 |0                          |
|events     |event_id            |0                 |0                          |
|order_items|order_id, product_id|110               |220                        |
|orders     |order_id            |0          

In [9]:
# Thống kê giá trị phân loại chính để phát hiện khoảng trắng thừa, chữ hoa chữ thường không thống nhất, hoặc giá trị lạ.
cac_cot_phan_loai = {
    "events": ["event_type", "payment"],
    "sessions": ["device", "source", "country"],
    "orders": ["payment_method", "country", "device", "source"],
    "customers": ["country", "marketing_opt_in"],
    "products": ["category"],
}

for ten_bang, cac_cot in cac_cot_phan_loai.items():
    bang_du_lieu = du_lieu_raw[ten_bang]
    for ten_cot in cac_cot:
        print(f"\n===== UNIQUE VALUES: {ten_bang}.{ten_cot} =====")
        (
            bang_du_lieu
            .groupBy(F.col(ten_cot).alias("gia_tri"))
            .agg(F.count("*").alias("so_luong"))
            .orderBy(F.desc("so_luong"), "gia_tri")
            .show(50, truncate=False)
        )


===== UNIQUE VALUES: events.event_type =====
+-----------+--------+
|gia_tri    |so_luong|
+-----------+--------+
|page_view  |539343  |
|add_to_cart|143126  |
|checkout   |44909   |
|purchase   |33580   |
+-----------+--------+


===== UNIQUE VALUES: events.payment =====
+-------+--------+
|gia_tri|so_luong|
+-------+--------+
|NULL   |727378  |
|card   |23455   |
|paypal |5032    |
|wallet |3373    |
|cod    |1720    |
+-------+--------+


===== UNIQUE VALUES: sessions.device =====
+-------+--------+
|gia_tri|so_luong|
+-------+--------+
|mobile |65942   |
|desktop|45547   |
|tablet |8511    |
+-------+--------+


===== UNIQUE VALUES: sessions.source =====
+--------+--------+
|gia_tri |so_luong|
+--------+--------+
|organic |40776   |
|direct  |29861   |
|paid    |14465   |
|social  |14389   |
|email   |10949   |
|referral|9560    |
+--------+--------+


===== UNIQUE VALUES: sessions.country =====
+-------+--------+
|gia_tri|so_luong|
+-------+--------+
|US     |21913   |
|IN     |9

## 4. Kiểm tra toàn vẹn quan hệ

Phần này kiểm tra khóa ngoại giữa các bảng. Với raw data, notebook tạo thêm `product_id_int` tạm thời từ `events.product_id` để kiểm tra quan hệ với `products.product_id`. Nếu dữ liệu processed đã tồn tại, notebook cũng kiểm tra lại trên processed data vì đây là lớp dữ liệu dùng cho Task 2 và Task 3.

In [10]:
# Chuẩn bị bảng events raw có thêm product_id_int tạm thời để kiểm tra join với products.
events_raw_cho_quan_he = du_lieu_raw["events"].withColumn("product_id_int", F.col("product_id").cast("int"))

# Đếm số khóa ngoại không khớp bằng left_anti join giữa bảng trái và bảng phải.
def dem_khong_khop(bang_trai, cot_trai, bang_phai, cot_phai, bo_qua_null=True):
    bang_trai_can_kiem_tra = bang_trai
    if bo_qua_null:
        bang_trai_can_kiem_tra = bang_trai_can_kiem_tra.filter(F.col(cot_trai).isNotNull())
    return (
        bang_trai_can_kiem_tra.alias("trai")
        .join(
            bang_phai.select(F.col(cot_phai).alias("khoa_phai")).distinct().alias("phai"),
            F.col(f"trai.{cot_trai}") == F.col("phai.khoa_phai"),
            "left_anti",
        )
        .count()
    )

# Kiểm tra quan hệ trên raw data để hiểu dữ liệu gốc trước khi cleaning.
ket_qua_quan_he_raw = [
    ("raw", "events.session_id -> sessions.session_id", dem_khong_khop(events_raw_cho_quan_he, "session_id", du_lieu_raw["sessions"], "session_id")),
    ("raw", "events.product_id_int -> products.product_id", dem_khong_khop(events_raw_cho_quan_he, "product_id_int", du_lieu_raw["products"], "product_id")),
    ("raw", "sessions.customer_id -> customers.customer_id", dem_khong_khop(du_lieu_raw["sessions"], "customer_id", du_lieu_raw["customers"], "customer_id")),
    ("raw", "orders.customer_id -> customers.customer_id", dem_khong_khop(du_lieu_raw["orders"], "customer_id", du_lieu_raw["customers"], "customer_id")),
    ("raw", "order_items.order_id -> orders.order_id", dem_khong_khop(du_lieu_raw["order_items"], "order_id", du_lieu_raw["orders"], "order_id")),
    ("raw", "order_items.product_id -> products.product_id", dem_khong_khop(du_lieu_raw["order_items"], "product_id", du_lieu_raw["products"], "product_id")),
]

spark.createDataFrame(
    ket_qua_quan_he_raw,
    ["lop_du_lieu", "quan_he", "unmatched_count"]
).show(truncate=False)

+-----------+---------------------------------------------+---------------+
|lop_du_lieu|quan_he                                      |unmatched_count|
+-----------+---------------------------------------------+---------------+
|raw        |events.session_id -> sessions.session_id     |0              |
|raw        |events.product_id_int -> products.product_id |0              |
|raw        |sessions.customer_id -> customers.customer_id|0              |
|raw        |orders.customer_id -> customers.customer_id  |0              |
|raw        |order_items.order_id -> orders.order_id      |0              |
|raw        |order_items.product_id -> products.product_id|0              |
+-----------+---------------------------------------------+---------------+



In [11]:
# Đọc bảng processed nếu Task 1 đã được chạy và thư mục processed tồn tại.
def doc_bang_processed(ten_thu_muc):
    duong_dan_bang = PROCESSED_DIR / ten_thu_muc
    if duong_dan_bang.exists():
        return spark.read.parquet(str(duong_dan_bang))
    return None

du_lieu_processed = {
    "events": doc_bang_processed("events_cleaned_parquet"),
    "customers": doc_bang_processed("customers_cleaned_parquet"),
    "sessions": doc_bang_processed("sessions_cleaned_parquet"),
    "products": doc_bang_processed("products_cleaned_parquet"),
    "orders": doc_bang_processed("orders_cleaned_parquet"),
    "order_items": doc_bang_processed("order_items_cleaned_parquet"),
}

# Nếu có đủ processed data, kiểm tra lại quan hệ trên lớp dữ liệu đã clean.
if all(bang_du_lieu is not None for bang_du_lieu in du_lieu_processed.values()):
    ket_qua_quan_he_processed = [
        ("processed", "events.session_id -> sessions.session_id", dem_khong_khop(du_lieu_processed["events"], "session_id", du_lieu_processed["sessions"], "session_id")),
        ("processed", "events.product_id_int -> products.product_id", dem_khong_khop(du_lieu_processed["events"], "product_id_int", du_lieu_processed["products"], "product_id")),
        ("processed", "sessions.customer_id -> customers.customer_id", dem_khong_khop(du_lieu_processed["sessions"], "customer_id", du_lieu_processed["customers"], "customer_id")),
        ("processed", "orders.customer_id -> customers.customer_id", dem_khong_khop(du_lieu_processed["orders"], "customer_id", du_lieu_processed["customers"], "customer_id")),
        ("processed", "order_items.order_id -> orders.order_id", dem_khong_khop(du_lieu_processed["order_items"], "order_id", du_lieu_processed["orders"], "order_id")),
        ("processed", "order_items.product_id -> products.product_id", dem_khong_khop(du_lieu_processed["order_items"], "product_id", du_lieu_processed["products"], "product_id")),
    ]
    spark.createDataFrame(
        ket_qua_quan_he_processed,
        ["lop_du_lieu", "quan_he", "unmatched_count"]
    ).show(truncate=False)
else:
    print("Chua co du processed data. Hay chay Task 1 truoc neu muon kiem tra quan he tren du lieu da clean.")

+-----------+---------------------------------------------+---------------+
|lop_du_lieu|quan_he                                      |unmatched_count|
+-----------+---------------------------------------------+---------------+
|processed  |events.session_id -> sessions.session_id     |0              |
|processed  |events.product_id_int -> products.product_id |0              |
|processed  |sessions.customer_id -> customers.customer_id|0              |
|processed  |orders.customer_id -> customers.customer_id  |0              |
|processed  |order_items.order_id -> orders.order_id      |0              |
|processed  |order_items.product_id -> products.product_id|0              |
+-----------+---------------------------------------------+---------------+



## 5. Phân tích phân phối và nghiệp vụ cơ bản

Phần này tạo một số insight ban đầu: hành vi người dùng theo `event_type`, khung giờ cao điểm, và thống kê sản phẩm theo category.

In [12]:
# Đếm số lượng event theo từng event_type để hiểu phân phối hành vi người dùng.
phan_phoi_event_type = (
    du_lieu_raw["events"]
    .groupBy("event_type")
    .agg(F.count("*").alias("so_luong_event"))
    .orderBy(F.desc("so_luong_event"))
)

phan_phoi_event_type.show(truncate=False)

+-----------+--------------+
|event_type |so_luong_event|
+-----------+--------------+
|page_view  |539343        |
|add_to_cart|143126        |
|checkout   |44909         |
|purchase   |33580         |
+-----------+--------------+



In [13]:
# Trích xuất giờ từ timestamp để tìm khung giờ có nhiều tương tác nhất.
phan_phoi_theo_gio = (
    du_lieu_raw["events"]
    .withColumn("thoi_gian_event", F.to_timestamp("timestamp"))
    .withColumn("gio_trong_ngay", F.hour("thoi_gian_event"))
    .groupBy("gio_trong_ngay")
    .agg(F.count("*").alias("so_luong_event"))
    .orderBy(F.desc("so_luong_event"))
)

phan_phoi_theo_gio.show(24, truncate=False)

+--------------+--------------+
|gio_trong_ngay|so_luong_event|
+--------------+--------------+
|22            |32365         |
|13            |32213         |
|18            |32141         |
|19            |32129         |
|10            |32055         |
|4             |31922         |
|21            |31921         |
|5             |31872         |
|6             |31850         |
|16            |31763         |
|14            |31749         |
|17            |31701         |
|9             |31597         |
|20            |31588         |
|0             |31542         |
|12            |31522         |
|1             |31489         |
|23            |31473         |
|11            |31473         |
|15            |31460         |
|3             |31436         |
|7             |31279         |
|8             |31254         |
|2             |31164         |
+--------------+--------------+



In [14]:
# Group products theo category để xem ngành hàng nào có nhiều sản phẩm và mức giá trung bình ra sao.
thong_ke_category = (
    du_lieu_raw["products"]
    .groupBy("category")
    .agg(
        F.count("product_id").alias("so_luong_san_pham"),
        F.round(F.avg("price_usd"), 2).alias("gia_trung_binh"),
        F.round(F.min("price_usd"), 2).alias("gia_thap_nhat"),
        F.round(F.max("price_usd"), 2).alias("gia_cao_nhat"),
    )
    .orderBy(F.desc("so_luong_san_pham"))
)

thong_ke_category.show(100, truncate=False)

+--------------+-----------------+--------------+-------------+------------+
|category      |so_luong_san_pham|gia_trung_binh|gia_thap_nhat|gia_cao_nhat|
+--------------+-----------------+--------------+-------------+------------+
|Home & Kitchen|171              |125.18        |10.91        |249.07      |
|Fashion       |171              |109.85        |9.69         |199.64      |
|Sports        |171              |154.19        |9.97         |299.44      |
|Electronics   |171              |315.57        |30.13        |596.62      |
|Books         |171              |26.21         |3.5          |49.97       |
|Beauty        |171              |64.66         |5.5          |119.56      |
|Toys          |171              |43.88         |6.51         |79.83       |
+--------------+-----------------+--------------+-------------+------------+



In [15]:
# Nếu có processed data, dùng lớp dữ liệu đã clean để xem nhanh các insight giống Task 2 và Task 3.
if du_lieu_processed.get("events") is not None:
    print("===== PHAN PHOI EVENT TYPE TREN PROCESSED DATA =====")
    (
        du_lieu_processed["events"]
        .groupBy("event_type")
        .agg(F.count("*").alias("so_luong_event"))
        .orderBy(F.desc("so_luong_event"))
        .show(truncate=False)
    )

    print("===== PEAK ACTIVITY THEO GIO TREN PROCESSED DATA =====")
    (
        du_lieu_processed["events"]
        .groupBy("event_hour")
        .agg(F.count("*").alias("so_luong_event"))
        .orderBy(F.desc("so_luong_event"))
        .show(24, truncate=False)
    )
else:
    print("Chua co events processed data de phan tich insight sau cleaning.")

===== PHAN PHOI EVENT TYPE TREN PROCESSED DATA =====
+-----------+--------------+
|event_type |so_luong_event|
+-----------+--------------+
|page_view  |539343        |
|add_to_cart|143126        |
|checkout   |44909         |
|purchase   |33580         |
+-----------+--------------+

===== PEAK ACTIVITY THEO GIO TREN PROCESSED DATA =====
+----------+--------------+
|event_hour|so_luong_event|
+----------+--------------+
|22        |32365         |
|13        |32213         |
|18        |32141         |
|19        |32129         |
|10        |32055         |
|4         |31922         |
|21        |31921         |
|5         |31872         |
|6         |31850         |
|16        |31763         |
|14        |31749         |
|17        |31701         |
|9         |31597         |
|20        |31588         |
|0         |31542         |
|12        |31522         |
|1         |31489         |
|23        |31473         |
|11        |31473         |
|15        |31460         |
|3         |314

## Kết luận nhanh

Sau khi chạy notebook, nên ghi lại các điểm sau vào báo cáo:

- Số dòng và số cột của từng bảng.
- Các cột có null nhiều nhất và lý do null trong `events`.
- Bảng nào có duplicate hoàn toàn hoặc duplicate khóa chính.
- Các quan hệ khóa ngoại có unmatched count hay không.
- Event type nào phổ biến nhất, giờ nào cao điểm nhất, category nào có nhiều sản phẩm nhất.

Nếu kết quả processed validation đều có unmatched count bằng 0, dữ liệu đã đủ tin cậy để tiếp tục Task 2 và Task 3.